# 01 · Data preparation — UCL 2026/27 dashboard

This notebook walks through `src/prepare.py` step by step: **load raw → show a problem → fix it → show the result**.
All logic lives in `src/`; the notebook only *calls* those functions, so the dashboard and this narrative always use the same code.

| Step | What | Function |
|---|---|---|
| 1 | Parse openfootball text files (2021-22 → 2025-26) | `parse_openfootball.parse_all` |
| 2 | Load football-data.org JSON (2026-27) | `prepare.load_footballdata` |
| 3 | Clean team & country names, combine, fix types | `team_names`, `prepare.combine` |
| 4 | Reshape wide → long (one row per team per match) | `prepare.to_team_matches` |
| 5 | Compute our own Elo ratings | `elo.compute_elo` |
| 6 | League table after every matchday | `prepare.league_tables` |
| 7 | How far each team got each season | `prepare.season_outcomes` |
| 8 | Current table + Elo, checked against the official table | `prepare.standings_2026` |
| 9 | Aggregate by country × season | `prepare.country_season` |
| 10 | Save to `data/processed/*.parquet` | |

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))          # so we can import our own modules

import prepare
from elo import compute_elo, expected_score, HOME_ADVANTAGE, K, START_RATING
from parse_openfootball import parse_all, parse_score
from team_names import TEAM_NAMES, country_from_code, league_country, map_teams

pd.set_option("display.max_columns", 30, "display.width", 140)
RAW_OF = ROOT / "data" / "raw" / "openfootball"
RAW_FD = ROOT / "data" / "raw" / "footballdata"

## 1 · openfootball: plain-text match files (2021-22 → 2025-26)

openfootball publishes results as human-readable text, not as a table. Below are a few raw lines.
Problems visible already: the **year is usually missing** from date lines, the **score has several shapes**, and the team name and country code are glued together (`Real Madrid CF (ESP)`).

In [2]:
lines = (RAW_OF / "2024-25" / "cl.txt").read_text(encoding="utf-8").splitlines()
print("\n".join(lines[:22]))

= UEFA Champions League 2024/25

# Date       Tue Sep 17 2024 - Sat May 31 2025 (256d)
# Teams      36
# Matches    189
# Stages     League (144)  Playoffs (16)  Finals (29)



▪ League, Matchday 1
  Tue Sep 17 2024
    18:45  BSC Young Boys (SUI)    v Aston Villa FC (ENG)     0-3 (0-2)
           Juventus FC (ITA)       v PSV (NED)                3-1 (2-0)
    21:00  Sporting Clube de Portugal (POR) v Lille OSC (FRA)          2-0 (1-0)
           FC Bayern München (GER) v GNK Dinamo Zagreb (CRO)  9-2 (3-0)
           Real Madrid CF (ESP)    v VfB Stuttgart (GER)      3-1 (0-0)
           AC Milan (ITA)          v Liverpool FC (ENG)       1-3 (1-2)
  Wed Sep 18
    18:45  Bologna FC 1909 (ITA)   v FK Shakhtar Donetsk (UKR)  0-0
           AC Sparta Praha (CZE)   v FC Red Bull Salzburg (AUT)  3-0 (2-0)
    21:00  Manchester City FC (ENG) v FC Internazionale Milano (ITA)  0-0
           Paris Saint-Germain FC (FRA) v Girona FC (ESP)          1-0 (0-0)


### Problem: four different score formats
The parser (`parse_score`) handles each shape with one regular expression and returns the final score, the 90-minute score, half-time, and penalties.
Penalties are kept separately because a shoot-out does not change the match result (it counts as a draw in UEFA statistics).

In [3]:
examples = ["2-1 (1-0)", "0-0", "3-1 a.e.t. (2-1, 0-0)", "4-3 pen. 1-1 a.e.t. (1-1, 0-1)"]
pd.DataFrame([{"raw": s, **parse_score(s)} for s in examples])

,raw,home_goals,away_goals,home_goals_90,away_goals_90,home_goals_ht,away_goals_ht,extra_time,home_pens,away_pens
0,2-1 (1-0),2,1,2,1,1,0,False,<NA>,<NA>
1,0-0,0,0,0,0,<NA>,<NA>,False,<NA>,<NA>
2,"3-1 a.e.t. (2-1, 0-0)",3,1,2,1,0,0,True,<NA>,<NA>
3,"4-3 pen. 1-1 a.e.t. (1-1, 0-1)",1,1,1,1,0,1,True,4,3


### Result: one tidy table
`parse_all()` reads every season into one DataFrame (one row = one match / one leg).
**Check:** each file has a header line `# Matches N`; the parser verifies that it found exactly N match lines, so no match was silently skipped.

In [4]:
of = prepare.load_openfootball()      # = parse_all() + status "FINISHED" (all historical matches were played)
print(f"openfootball: {len(of)} matches, {of['season'].nunique()} seasons")
of.groupby(["season", "stage"]).size().unstack(fill_value=0)

openfootball: 753 matches, 5 seasons


stage,Finals,Group,League,Playoffs
season,,,,
2021-22,29,96,0,0
2022-23,29,96,0,0
2023-24,29,96,0,0
2024-25,29,0,144,16
2025-26,29,0,144,16


Two competition formats are visible: **2021-24** had 8 groups of 4 (96 group matches), **2024-26** the new 36-team league phase (144 matches + 16 play-off legs).
That is why Q2 (early form) uses only 2024-25 and 2025-26.

## 2 · football-data.org: the current season (2026-27)

openfootball has no 2026-27 file yet, so the current season comes from the football-data.org API (free tier, JSON).
One raw match record:

In [5]:
raw_fd = json.loads((RAW_FD / "cl_2026_matches.json").read_text(encoding="utf-8"))["matches"]
m = raw_fd[0]
{k: m[k] for k in ["utcDate", "status", "matchday", "stage"]} | {
    "home": m["homeTeam"]["name"], "away": m["awayTeam"]["name"], "fullTime": m["score"]["fullTime"]}

{'utcDate': '2026-09-08T16:45:00Z',
 'status': 'FINISHED',
 'matchday': 1,
 'stage': 'LEAGUE_STAGE',
 'home': 'Club Brugge KV',
 'away': 'Aston Villa FC',
 'fullTime': {'home': 2, 'away': 3}}

In [6]:
fd = prepare.load_footballdata()
print(f"football-data: {len(fd)} matches")
fd.groupby(["matchday", "status"]).size().unstack(fill_value=0)

football-data: 144 matches


status,FINISHED,TIMED
matchday,,
1,18,0
2,18,0
3,18,0
4,18,0
5,0,18
6,0,18
7,0,18
8,0,18


**Decision — unplayed matches are kept, not dropped.** Matchdays 5–8 are scheduled (`TIMED`) but not played, so their goals are missing (`NaN`).
They are kept with `played = False` because the Monte Carlo simulation (Q4) needs the remaining fixtures. Every calculation of points, Elo and tables filters to played matches only.

## 3 · Cleaning team and country names, then combining

### Problem: the same club has several spellings
Even *inside* openfootball, names change between seasons. Without cleaning, "Real Madrid" and "Real Madrid CF" would count as two clubs and break every per-team chart.

In [7]:
raw_names = pd.concat([of["home_team"], of["away_team"], fd["home_team"], fd["away_team"]])
spellings = (pd.DataFrame({"raw": raw_names.unique()})
             .assign(canonical=lambda d: d["raw"].map(TEAM_NAMES)))
multi = spellings.groupby("canonical")["raw"].apply(list)
multi[multi.str.len() > 1].head(10)

canonical
Atalanta                              [Atalanta, Atalanta BC]
Atlético Madrid    [Atlético Madrid, Club Atlético de Madrid]
Bayern Munich             [Bayern München, FC Bayern München]
Benfica                  [SL Benfica, Sport Lisboa e Benfica]
Dinamo Zagreb              [Dinamo Zagreb, GNK Dinamo Zagreb]
Inter                       [Inter, FC Internazionale Milano]
Juventus                              [Juventus, Juventus FC]
Leverkusen            [Bayer Leverkusen, Bayer 04 Leverkusen]
Man City                [Manchester City, Manchester City FC]
Man United          [Manchester United, Manchester United FC]
Name: raw, dtype: object

### Fix: one explicit mapping table (`src/team_names.py`)
Every raw spelling is listed by hand — **no fuzzy matching**, so each mapping can be checked by eye. Canonical names are short (`PSG`, `Man City`) because they are used as chart labels.
An unknown spelling **raises an error** instead of being silently dropped:

In [8]:
print(f"{spellings['raw'].nunique()} raw spellings -> {spellings['canonical'].nunique()} canonical clubs")
try:
    map_teams(pd.Series(["Real Madrid CF", "Real Madird"]))     # deliberate typo
except KeyError as e:
    print("Error raised as intended:", e)

97 raw spellings -> 80 canonical clubs
Error raised as intended: "Unmapped team names (add to src/team_names.py): ['Real Madird']"


### Problem: countries are coded differently
openfootball uses 3-letter codes (`ENG`), football-data full names (`England`). We convert codes to names with `country_from_code` (again: unknown code → error).
For Q3 (*national leagues*) AS Monaco is counted as **France**, because it plays in Ligue 1 — that is `league_country`.

In [9]:
print(country_from_code("ENG"), "|", country_from_code("ESP"), "|", "Monaco ->", league_country("Monaco"))

England | Spain | Monaco -> France


### Combine and fix data types
`combine()` maps all names, stacks both sources, and converts types:
goal columns become **nullable integers** (`Int64`) — plain `int` cannot hold "missing" for unplayed matches; `date` becomes a real datetime; and two derived flags are added: `neutral` (only the final is at a neutral venue, relevant for Elo home advantage) and `played`.

In [10]:
matches = prepare.combine(of, fd)
print(f"before: {len(of)} openfootball + {len(fd)} football-data = {len(of) + len(fd)}")
print(f"after : {len(matches)} rows  (no rows dropped), {matches['played'].sum()} played")
matches[["date", "home_goals", "home_goals_ht", "home_pens", "extra_time", "played"]].dtypes

before: 753 openfootball + 144 football-data = 897
after : 897 rows  (no rows dropped), 825 played


date             datetime64[us]
home_goals                Int64
home_goals_ht             Int64
home_pens                 Int64
extra_time                 bool
played                     bool
dtype: object

### Missing-values summary
Every missing value has a reason; none needs imputing:

In [11]:
na = matches.isna().sum()
reasons = {
    "group": "only 2021-24 group stage has groups",
    "matchday": "knockout rounds and 2021-23 group files have no matchday number",
    "home_goals": "2026-27 matchdays 5-8 not played yet", "away_goals": "same",
    "home_goals_90": "same", "away_goals_90": "same",
    "home_goals_ht": "unplayed + some openfootball lines give no half-time score",
    "away_goals_ht": "same",
    "home_pens": "only matches decided by a shoot-out have penalties", "away_pens": "same",
}
pd.DataFrame({"missing": na[na > 0], "reason": pd.Series(reasons)}).dropna(subset=["missing"]).astype({"missing": int})

,missing,reason
group,705,only 2021-24 group stage has groups
matchday,337,knockout rounds and 2021-23 group files have n...
home_goals,72,2026-27 matchdays 5-8 not played yet
away_goals,72,same
home_goals_90,72,same
away_goals_90,72,same
home_goals_ht,112,unplayed + some openfootball lines give no hal...
away_goals_ht,112,same
home_pens,891,only matches decided by a shoot-out have penal...
away_pens,891,same


**Decision — missing half-time scores are left missing.** Half-time goals are not used in any chart, so guessing them would add invented data for no benefit.

## 4 · Reshape: wide → long

`matches` is *wide* (home and away in separate columns). Questions like "how many points does Arsenal have?" would need two lookups.
`to_team_matches()` turns each played match into **two rows** — one per team — and derives `goal_diff`, `result` (W/D/L) and `points` (3/1/0).
A penalty shoot-out counts as a draw (result after extra time), as in official UEFA statistics.

In [12]:
team_matches = prepare.to_team_matches(matches)
print(f"{matches['played'].sum()} played matches x 2 = {len(team_matches)} team-match rows")
team_matches[team_matches["season"].eq("2026-27")].head(4)

825 played matches x 2 = 1650 team-match rows


,season,stage,round,matchday,date,team,opponent,country,venue,goals_for,goals_against,league_country,goal_diff,result,points
1506,2026-27,League,Matchday 1,1,2026-09-08,AEK Athens,LASK,Greece,home,1,0,Greece,1,W,3
1507,2026-27,League,Matchday 1,1,2026-09-08,Aston Villa,Club Brugge,England,away,3,2,England,1,W,3
1508,2026-27,League,Matchday 1,1,2026-09-08,Club Brugge,Aston Villa,Belgium,home,2,3,Belgium,-1,L,0
1509,2026-27,League,Matchday 1,1,2026-09-08,Dortmund,Villarreal,Germany,home,3,2,Germany,1,W,3


## 5 · Our own Elo ratings

**Why:** ClubElo's API returned HTTP 502 for over a week, so we apply the same standard Elo method to our own match data (`src/elo.py`).

- Before each match: `expected = 1 / (1 + 10^(-(R_home + home_adv - R_away) / 400))`
- After: `R_new = R_old + K × (actual − expected)` with actual = 1 / 0.5 / 0 — the winner takes points from the loser, more for a surprise.
- Choices: start rating **1500**, **K = 20** (as ClubElo), home advantage **50** points (none in the final), shoot-out = draw.

**Limitation:** only UCL matches (8–17 per team per season), so ratings are noisier than ClubElo's; 2021-22 is a warm-up season in which every club starts equal.

In [13]:
# Worked example: a 1600 team at home vs a 1500 team
e = expected_score(1600 + HOME_ADVANTAGE, 1500)
print(f"expected score = {e:.2f}; if home wins: +{K * (1 - e):.1f}, if home loses: {K * (0 - e):.1f} Elo points")

expected score = 0.70; if home wins: +5.9, if home loses: -14.1 Elo points


In [14]:
elo_hist = compute_elo(matches[matches["played"]])
elo_now = elo_hist.sort_values("date").groupby("team")["elo_after"].last()
print(f"{len(elo_hist)} rating updates (= team-match rows: {len(elo_hist) == len(team_matches)})")
elo_now.sort_values(ascending=False).head(10).round(0).to_frame("current Elo")

1650 rating updates (= team-match rows: True)


,current Elo
team,
Bayern Munich,1683.0
PSG,1677.0
Arsenal,1671.0
Real Madrid,1669.0
Liverpool,1616.0
Man City,1612.0
Barcelona,1608.0
Inter,1590.0
Dortmund,1571.0


**Sanity check:** the top of the list (Bayern, PSG, Arsenal, Real Madrid) are the clubs that have gone deep in recent seasons — the ratings behave sensibly.

## 6 · League table after every matchday

For the bump chart and Q2 we need the standings **after each matchday**, which no source provides. `league_tables()` cumulates points per team and ranks them by UEFA's first five tie-breakers
(points, goal difference, goals scored, away goals scored, wins); a remaining tie is broken alphabetically.
Built in-code: every (season, matchday) must contain exactly 36 teams.

In [15]:
tables = prepare.league_tables(team_matches)
print(tables.groupby("season")["matchday"].max().rename("matchdays available").to_string())
tables.query("season == '2024-25' and matchday == 8").head(8)[["position", "team", "points", "goal_diff", "goals_for"]]

season
2024-25    8
2025-26    8
2026-27    4


,position,team,points,goal_diff,goals_for
252,1,Liverpool,21,12,17
253,2,Barcelona,19,15,28
254,3,Arsenal,19,13,16
255,4,Inter,19,10,11
256,5,Atlético Madrid,18,8,20
257,6,Leverkusen,16,8,15
258,7,Lille,16,7,17
259,8,Aston Villa,16,7,13


**Check against reality:** the 2024-25 final top 8 (Liverpool, Barcelona, Arsenal, Inter, Atlético, Leverkusen, Lille, Aston Villa) matches the official UEFA table.

## 7 · Season outcomes: how far did each team get?

Knockout ties have two legs and openfootball gives no aggregate. Instead of recomputing aggregates (extra time, penalties…), we use a simpler rule:
**a team that plays in the next round has advanced.** Only the single-match final needs a rule (goals, then penalties).
Binary flags (`reached_r16`, `reached_sf`, `won_title`…) make Q2 a simple mean.

In [16]:
outcomes = prepare.season_outcomes(matches, team_matches, tables)
outcomes[outcomes["won_title"]][["season", "team", "country"]]

,season,team,country
22,2021-22,Real Madrid,Spain
50,2022-23,Man City,England
88,2023-24,Real Madrid,Spain
119,2024-25,PSG,France
158,2025-26,PSG,France


**Check:** the five champions (Real Madrid, Man City, Real Madrid, PSG, PSG) are correct.

In [17]:
# Preview of Q2: does position after matchday 4 predict a deep run?
q2 = outcomes.query("season in ['2024-25', '2025-26']").copy()
q2["md4_band"] = pd.cut(q2["position_md4"], [0, 8, 16, 24, 36], labels=["1-8", "9-16", "17-24", "25-36"])
q2.groupby("md4_band", observed=True)[["reached_r16", "reached_qf", "reached_sf"]].mean().round(2)

,reached_r16,reached_qf,reached_sf
md4_band,,,
1-8,0.75,0.56,0.31
9-16,0.56,0.19,0.06
17-24,0.56,0.19,0.06
25-36,0.08,0.04,0.04


## 8 · Current standings + Elo (Q1, Q4)

Our computed 2026-27 table is **checked against the official API table**: points must be identical (assertion), positions differing only on deep tie-breakers would be reported.
`rank_gap = elo_rank − position`: positive means a team sits higher in the table than its Elo suggests (over-performing).

In [18]:
standings = prepare.standings_2026(tables, elo_now)
standings.head(10)

,position,team,league_country,played,points,goal_diff,goals_for,won,form,elo,elo_rank,rank_gap
0,1,PSG,France,4,12,15,20,4,"W,W,W,W",1676.694834,2,1
1,2,Real Madrid,Spain,4,12,6,13,4,"W,W,W,W",1668.845356,4,2
2,3,Arsenal,England,4,10,5,8,3,"W,D,W,W",1671.422953,3,0
3,4,Como,Italy,4,9,13,19,3,"W,W,L,W",1517.677607,15,11
4,5,Barcelona,Spain,4,9,8,14,3,"W,L,W,W",1608.281530,7,2
5,6,Aston Villa,England,4,9,6,12,3,"L,W,W,W",1561.197685,10,4
6,7,Man City,England,4,9,5,12,3,"W,W,L,W",1611.626463,6,-1
7,8,Liverpool,England,4,9,4,14,3,"L,W,W,W",1615.878317,5,-3
8,9,Club Brugge,Belgium,4,9,4,11,3,"W,W,W,L",1506.447205,16,7
9,10,Sporting CP,Portugal,4,9,4,8,3,"W,L,W,W",1532.174480,12,2


In [19]:
cols = ["position", "team", "points", "elo_rank", "rank_gap"]
print("Most over-performing:");  print(standings.nlargest(3, "rank_gap")[cols].to_string(index=False))
print("\nMost under-performing:"); print(standings.nsmallest(3, "rank_gap")[cols].to_string(index=False))

Most over-performing:
 position             team  points  elo_rank  rank_gap
       21 Shakhtar Donetsk       4        33        12
        4             Como       9        15        11
       17       Man United       6        28        11

Most under-performing:
 position            team  points  elo_rank  rank_gap
       35           Lille       1        21       -14
       13   Bayern Munich       8         1       -12
       23 Atlético Madrid       4        11       -12


## 9 · Aggregate by country × season (Q3)

Two kinds of aggregation joined into one table: **per club** (number of clubs, mean/best end-of-season Elo, clubs reaching R16/QF, titles) and **per match** (win rate, points and goal difference per match).
`season_complete` flags 2026-27 so the dashboard can label it as in progress.

In [20]:
countries = prepare.country_season(team_matches, outcomes, elo_hist)
countries.pivot_table(index="league_country", columns="season", values="points_per_match").round(2) \
    .loc[["England", "Spain", "Germany", "Italy", "France"]]

season,2021-22,2022-23,2023-24,2024-25,2025-26,2026-27
league_country,,,,,,
England,2.05,1.85,1.62,1.93,1.90,2.15
Spain,1.49,1.43,1.79,1.67,1.43,1.85
Germany,1.57,1.62,1.62,1.50,1.57,1.69
Italy,1.39,1.63,1.57,1.60,1.38,1.38
France,1.56,1.43,1.39,1.66,1.57,1.33


## 10 · Save to `data/processed/`

Parquet keeps the data types (nullable integers, dates, ordered categories) that CSV would lose. The dashboard reads **only** these files.

In [21]:
outputs = {"matches": matches, "team_matches": team_matches, "elo_history": elo_hist,
           "league_tables": tables, "season_outcomes": outcomes,
           "standings_2026": standings, "country_season": countries}
OUT = ROOT / "data" / "processed"
for name, df in outputs.items():
    df.to_parquet(OUT / f"{name}.parquet", index=False)
pd.DataFrame({"rows": {n: len(d) for n, d in outputs.items()},
              "columns": {n: d.shape[1] for n, d in outputs.items()}})

,rows,columns
matches,897,23
team_matches,1650,15
elo_history,1650,8
league_tables,720,11
season_outcomes,204,13
standings_2026,36,12
country_season,93,13


## Cleaning-decision log

| Decision | Why |
|---|---|
| No rows dropped; 72 unplayed 2026-27 matches kept with `played=False` | needed for the Q4 simulation; filtered out of every statistic |
| Missing half-time scores left missing | not used anywhere; imputing would invent data |
| 97 distinct raw spellings → 80 clubs via explicit table; unknown name = error | one club must never count as two; nothing silently lost |
| Country codes → full names; Monaco → France for league analysis | two sources use different country formats; Monaco plays in Ligue 1 |
| Penalty shoot-out = draw for results and Elo | matches UEFA statistics; shoot-outs say little about strength |
| Knockout progress = "appears in next round" | avoids re-implementing aggregate/extra-time rules; verified with the 5 champions |
| Own Elo (1500 / K=20 / home +50) | ClubElo API down; limitation: UCL matches only |
| Our 2026-27 table checked against the API table | proves the cumulative-table logic is correct |